## Plan

**First run test plan**

0. Add/patch apply_variant in ecoli/variants/gene_knockout.py


The 5 successful variant/seed pairs avoid this because they never reach a timestep where `mRNA_unique_index` and the ribosome-derived `unique_mRNA_index_ribosomes` are both empty. In other words, there is still at least one full, translatable mRNA left for the listener to map, so `bulk_name_to_idx(...)` has valid inputs.
1. Dry-run: create variant pickles only (use create_variants.py)
```
python runscripts/create_variants.py --config configs/N_gene_knockout_test.json --kb out/all_media_conditions1/parca/kb -o out/test_variants
```

2. Check out/test_variants/metadata.json and open one .cPickle to confirm sim_data.genetic_perturbations or adjusted arrays were set.

3. Run full workflow (will launch Nextflow)
(If Nextflow/containers not configured, run on a machine with Nextflow installed. Use --resume to restart.)
```
python runscripts/workflow.py --config configs/N_gene_knockout_test.json
```

4. Analyze with gene_screen.py / gene_expression_trace.py after sims finish:
(or run gene_expression_trace.py for a specific gene)
```
python reading/gene_screen.py --project gene_knockout_test --variants 1 2 --generations 1 --gene-list /user/home/il22158/work/vEcoli/reading/results/knockout_experiment/knocked_out_test.txt
```

**Second run test plan**
1. Same procedure as the first run test step 3 and 4, with the problematic gene list (p_lilst) from previous simulation.
2. Run gene screen with the p_list.
3. Without knowing how many of variants run with success status:
```bash
python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list /user/home/il22158/work/vEcoli/surrogate/results/failure/outlier_genes_all_unique.txt
```
4. After knowing how many of variants run with success status:
```bash
python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants 0 1 2 47 48  --gene-list /user/home/il22158/work/vEcoli/surrogate/results/failure/outlier_genes_all_unique.txt
```

**Third test - simulation of successful run list**
1. Same procedure as the second test (aggregate_knockout_results.py), only difference is that the gene list are the ones which succeeds to 8th in previous run.
- results seems wrong, some simulation results are missing while valid in out folder
2. Current - most of them  (46 out of 50) succeeds to 8th generation in the new simulation.
3. Further anlysis for gene expression - done
```bash
python reading/gene_screen.py --project gene_knockout_3_round_test --lineage-seed 100 101 --variants 0 1 2 47 48  --gene-list /user/home/il22158/work/vEcoli/surrogate/third_round_tested_gene_list.txt
```

**Forth test - knockout with RNA/TU IDs**
1. Functional gene ids is converted to RNA/TU ids to aovid duplicate runs, sored in [test_tu_ids_to_test.csv](/user/home/il22158/work/vEcoli/surrogate/rest_tu_ids_to_test.csv).
2. The first RNA id and the first TU id is picked to run the simulation, see [N_gene_knockout_TU_ID_test.json](/user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_test.json).
3. Simulation executated by workflow.py wtih baseline media, two seeds, and 8 generations.
4. Read the output for both a) gene expression, and b) growth rate.


## Test1 - Variant creation 2 knockouts test

In [11]:
import pickle
import numpy as np
import os

base_dir = '/user/home/il22158/work/vEcoli/out/test_variants'

with open(os.path.join(base_dir, '0.cPickle'), 'rb') as f:
    sd_v0 = pickle.load(f)
with open(os.path.join(base_dir, '1.cPickle'), 'rb') as f:
    sd_v1 = pickle.load(f)
with open(os.path.join(base_dir, '2.cPickle'), 'rb') as f:
    sd_v2 = pickle.load(f)

transcription_v0 = sd_v0.process.transcription
transcription_v1 = sd_v1.process.transcription
transcription_v2 = sd_v2.process.transcription

def analyze_variant(v0, v1, variant_name):
    """Compare two transcription objects and return metrics"""
    print("="*100)
    print(f"PARAMETER COMPARISON: {variant_name}")
    print("="*100)

    metrics = {}

    # ============================================================================
    # 1. rna_synth_prob
    # ============================================================================
    print("\n1. rna_synth_prob (dict of arrays)")
    print("-" * 100)

    rna_synth_prob_v0 = v0.rna_synth_prob
    rna_synth_prob_v1 = v1.rna_synth_prob

    keys_v0 = set(rna_synth_prob_v0.keys())
    keys_v1 = set(rna_synth_prob_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_synth_prob_v0.keys():
        arr_v0 = rna_synth_prob_v0[key]
        arr_v1 = rna_synth_prob_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_synth_prob_v0['basal'])
    basal_sum_v1 = np.sum(rna_synth_prob_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_synth_prob'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 2. rna_expression
    # ============================================================================
    print("\n2. rna_expression (dict of arrays)")
    print("-" * 100)

    rna_expression_v0 = v0.rna_expression
    rna_expression_v1 = v1.rna_expression

    keys_v0 = set(rna_expression_v0.keys())
    keys_v1 = set(rna_expression_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_expression_v0.keys():
        arr_v0 = rna_expression_v0[key]
        arr_v1 = rna_expression_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_expression_v0['basal'])
    basal_sum_v1 = np.sum(rna_expression_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_expression'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 3. exp_free
    # ============================================================================
    print("\n3. exp_free (1D array)")
    print("-" * 100)

    exp_free_v0 = v0.exp_free
    exp_free_v1 = v1.exp_free

    shape_v0 = exp_free_v0.shape
    shape_v1 = exp_free_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_free_v0 - exp_free_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_free_v0)
    sum_v1 = np.sum(exp_free_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_free'] = {'max_diff': max_diff, 'sum': sum_v1}

    # ============================================================================
    # 4. exp_ppgpp
    # ============================================================================
    print("\n4. exp_ppgpp (1D array)")
    print("-" * 100)

    exp_ppgpp_v0 = v0.exp_ppgpp
    exp_ppgpp_v1 = v1.exp_ppgpp

    shape_v0 = exp_ppgpp_v0.shape
    shape_v1 = exp_ppgpp_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_ppgpp_v0 - exp_ppgpp_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_ppgpp_v0)
    sum_v1 = np.sum(exp_ppgpp_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_ppgpp'] = {'max_diff': max_diff, 'sum': sum_v1}
    
    return metrics

# ============================================================================
# Run analysis for Variant 1 (EG10109 KO)
# ============================================================================
metrics_v1 = analyze_variant(transcription_v0, transcription_v1, "Variant 1 (EG10109 KO) vs Baseline")

# ============================================================================
# Run analysis for Variant 2 (EG10196 KO)
# ============================================================================
print("\n\n")
metrics_v2 = analyze_variant(transcription_v0, transcription_v2, "Variant 2 (EG10196 KO) vs Baseline")

# ============================================================================
# Summary Table - Both Variants Side-by-Side
# ============================================================================
print("\n\n" + "="*130)
print("FINAL TABLE: Parameter Changes After Gene Knockout (Both Variants)")
print("="*130)
print(f"{'Parameter':<20} {'Variant 1 (EG10109 KO)':<55} {'Variant 2 (EG10196 KO)':<55}")
print(f"{'':20} {'Structure':<15} {'Values':<15} {'Sum':<15} {'Structure':<15} {'Values':<15} {'Sum':<15}")
print("-" * 130)

params = ['rna_synth_prob', 'rna_expression', 'exp_free', 'exp_ppgpp']
structures = {
    'rna_synth_prob': '✓ Keys identical',
    'rna_expression': '✓ Keys identical',
    'exp_free': '✓ Shape (3277,)',
    'exp_ppgpp': '✓ Shape (3277,)'
}

for param in params:
    v1_max_diff = metrics_v1[param]['max_diff']
    v1_sum = metrics_v1[param]['sum']
    v2_max_diff = metrics_v2[param]['max_diff']
    v2_sum = metrics_v2[param]['sum']
    
    struct = structures[param]
    v1_values = f"✗ Δ {v1_max_diff:.2e}"
    v2_values = f"✗ Δ {v2_max_diff:.2e}"
    v1_sum_str = f"✓ {v1_sum:.6f}"
    v2_sum_str = f"✓ {v2_sum:.6f}"
    
    print(f"{param:<20} {struct:<15} {v1_values:<15} {v1_sum_str:<15} {struct:<15} {v2_values:<15} {v2_sum_str:<15}")

print("="*130)


PARAMETER COMPARISON: Variant 1 (EG10109 KO) vs Baseline

1. rna_synth_prob (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 2.73e-05
Sum of 'basal' array: V0=1.000000, V1=1.000000

2. rna_expression (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 3.85e-06
Sum of 'basal' array: V0=1.000000, V1=1.000000

3. exp_free (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.51e-06
Sum of array: V0=1.0000000000, V1=1.0000000000

4. exp_ppgpp (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.89e-06
Sum of array: V0=1.00000000

- Conclusion:
    The modification of gene expression and systematic renormalization work at the variant creation process.

## Test1 - 8 Generations 2 Knockouts test

- Inputs:
    Baseline - no knockouts;
    Variant 1 - knockout gene EG10109 (Gene 1);
    Variant 2 - knockout gene EG10196 (Gene 2).

- Results:
    Gene screen for two knockout genes saved in:
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_0
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_1
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_2

    Where we can see the results matches with expectation - as the two genes both expressed (translated) in baseline, while only Gene 2 expressed in Varaint 1 and only Gene 1 expressed in Variant 2.

    Meanwhile, the cell fate also matched with the early simulation (cutomised knockout function) - Variant 1 succeed to generation 8 and Variant 2 doesn't.

    The main difference between tested knockouts and early simulation is that the Variant 2 cell dies sooner - it only succeeds to 3rd generation while the early simulation succeeds to 6th generation.

## Test2 - Read outputs in p_list KO sim

**Plan for p_list KO Simulation Analysis**

1. **Run multi-seed gene screening** (seeds 100 and 101 across 52 variants)
   - Command: `python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list outlier_genes_all_unique.txt`
   - Expected output structure: `/out/gene_knockout_p_list/{project}/{variant}/{lineage_seed}/`
   - Outputs: CSV files with gene activity, transcription/translation traces, and plots

2. **Aggregate results** - Run aggregation script after screening completes
   - Command: `python reading/aggregate_knockout_results.py --project gene_knockout_p_list`
   - Optional: add `--output-dir`, `--output-file`, or `--base-path` to override defaults
   - Purpose: Consolidate all 52 variants × 2 seeds results into a single summary CSV
   - Input: Success files from `/out/gene_knockout_p_list/success/experiment_id=gene_knockout_p_list/...`
   - Output (default): `/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv`
   - Format: Similar to existing `ko_runs_with_success_generation.csv` with columns:
     * project, variant, lineage_seed, max_success_generation, duration_sec, final_dry_mass, growth_rate, etc.

3. **Validation & analysis**
   - Verify no figure overlap between seeds
   - Check all 52 variants processed with both seeds (expected: 104 result rows)
   - Confirm CSV format matches reference standard
   - Compare seed 100 vs seed 101 results statistically


## Understanding Test 2 results (Debug?)

**Investigation Summary**

The knockout definition itself looks reasonable. In [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py), `apply_variant()` maps each `genes_to_knockout` entry to cistron/RNA indices and calls `sim_data.adjust_final_expression(...)` only when it finds a match. The attached config [configs/N_gene_knockout_p_list.json](/user/work/il22158/vEcoli/configs/N_gene_knockout_p_list.json) is also consistent with that behavior: it applies the `gene_knockout` variant to a 52-gene list and runs two seeds (`100` and `101`).

Error Message Location: [nextflow command log](/user/home/il22158/work/vEcoli/out/gene_knockout_p_list/nextflow/nextflow_workdirs/02/db2dbbe190b7231220777600953584/.command.log)

The failure is downstream in the simulation listener, not in the variant generator. A failed task in the Nextflow work directory showed this traceback from [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py):

`IndexError: cannot do a non-empty take from an empty axes.`

That happens at the `bulk_name_to_idx(...)` call where `reduced_to_normal_mRNA_indices` is built. For the failing variants/seeds, the mRNA index arrays are empty enough that `np.take(...)` cannot proceed.

The aggregation output [surrogate/results/gene_knockout_p_list_success_summary.csv](/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv) confirms this is not a summary bug: it contains only 10 successful rows, meaning 5 variant/seed combinations completed successfully while the rest failed earlier in the workflow.

The successful variants do not hit this because they still leave at least one full, translatable mRNA in the listener state when `ribosome_data.py` runs. In that case `mRNA_unique_index` and `unique_mRNA_index_ribosomes` are non-empty, so `bulk_name_to_idx(...)` has valid inputs and never reaches the empty-axis path.

Potential fix: Don't change [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py) as the primary fix. The safer change is to harden [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py) against empty mRNA/ribosome index arrays and return zero-filled listener values instead of raising. A small defensive check there should prevent the crash for knockout cases that leave no valid mRNA targets.



## Test3 - Comparison Run: sucessful run gene list

**Third Round Test: gene knockout list for variants that looks successful in previous simulation**

**To-do plan**

1. Prepare the third-round knockout input from the successful generation-8 gene set.
2. Run the simulation with renewed configuration.
3. Run the knockout screening with the same project structure and the new gene list file.
   - Command: `python reading/gene_screen.py --project gene_knockout_3_round_test --lineage-seed 100 101 --variants $(seq 0 50) --gene-list surrogate/third_round_tested_gene_list.txt`
4. Aggregate the results and confirm which variants reach generation 8 again. (results: [gene_knockout_3_round_test_success_summary.csv](/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_3_round_test_success_summary.csv))
5. Compare this round against the previous run to see whether the success pattern is reproducible.

**The tested gene list**

They are extracted as the first 50 gene knockouts which does not show unexpected growth rate raise pattern in previous simulation (with customised KO function).

Link: [third_round_tested_gene_list.txt](/user/home/il22158/work/vEcoli/surrogate/third_round_tested_gene_list.txt).

In [4]:
import pandas as pd
from pathlib import Path

# Load the test3 results
base = "/user/home/il22158/work/vEcoli/"
test3_csv = Path(f"{base}surrogate/results/gene_knockout_3_round_test_success_summary.csv")
gene_list_file = Path(f"{base}surrogate/third_round_tested_gene_list.txt")
previous_csv = Path(f"{base}surrogate/results/ko_runs_with_success_generation.csv")

# Read test3 results
test3_df = pd.read_csv(test3_csv)

# Read gene list (variant to gene mapping)
# Note: variant 0 is baseline, variants 1-50 correspond to gene indices 0-49
with open(gene_list_file) as f:
    genes = [line.strip() for line in f if line.strip()]

# Create mapping: variant -> gene (accounting for variant 0 being baseline)
variant_to_gene = {}
variant_to_gene[0] = "BASELINE"  # Variant 0 is baseline (no KO)
for idx, gene in enumerate(genes):
    variant_to_gene[idx + 1] = gene  # Variant i (i>=1) maps to gene at index i-1

# Read previous simulation results
previous_df = pd.read_csv(previous_csv)

# Extract gene IDs from label column (format: "KO: EG10109")
previous_df['gene_from_label'] = previous_df['label'].str.replace('KO: ', '').str.strip()

# Create comparison results - one row per seed per variant
results = []
for variant in sorted(set(test3_df['variant'])):
    gene = variant_to_gene.get(variant, "UNKNOWN")
    
    # Get test3 results for this variant
    test3_var = test3_df[test3_df['variant'] == variant]
    
    # Get previous results for this gene (match by gene name, skip baseline)
    if gene == "BASELINE":
        previous_var = pd.DataFrame()  # No baseline in previous results
    else:
        previous_var = previous_df[previous_df['gene_from_label'] == gene]
    
    # Process each seed
    for seed in [100, 101]:
        # Test3 data for this seed
        test3_seed_data = test3_var[test3_var['lineage_seed'] == seed]
        test3_gen = test3_seed_data['max_success_generation'].values[0] if len(test3_seed_data) > 0 else None
        test3_reaches_gen8 = (test3_gen == 8) if test3_gen is not None else None
        
        # Previous data for this seed
        if len(previous_var) > 0:
            previous_seed_data = previous_var[previous_var['lineage_seed'] == seed]
            previous_gen = previous_seed_data['max_success_generation'].values[0] if len(previous_seed_data) > 0 else None
            previous_reaches_gen8 = (previous_gen == 8) if previous_gen is not None else None
        else:
            previous_gen = None
            previous_reaches_gen8 = None
        
        # Check if they match (only if both have data)
        if test3_reaches_gen8 is not None and previous_reaches_gen8 is not None:
            reaches_gen8_match = (test3_reaches_gen8 == previous_reaches_gen8)
            max_gen_match = (test3_gen == previous_gen)
        else:
            reaches_gen8_match = False  # Mark as non-match if data is missing
            max_gen_match = False
        
        results.append({
            'variant': variant,
            'gene': gene,
            'seed': seed,
            'test3_reaches_gen8': test3_reaches_gen8,
            'previous_reaches_gen8': previous_reaches_gen8,
            'reaches_gen8_match': reaches_gen8_match,
            'test3_max_gen': int(test3_gen) if test3_gen else None,
            'previous_max_gen': int(previous_gen) if previous_gen else None,
            'max_gen_match': max_gen_match
        })

results_df = pd.DataFrame(results)

# Save results
output_dir = Path(f"{base}surrogate/results/test3")
output_dir.mkdir(exist_ok=True)
output_csv = output_dir / "test3_comparison_results.csv"
results_df.to_csv(output_csv, index=False)

print(f"✓ Comparison results saved to: {output_csv}")
print(f"\nSummary:")
total_variants = len(set(results_df['variant']))
print(f"Total variants tested: {total_variants}")
print(f"Rows (variants × seeds): {len(results_df)}")

# Summary of matches
reach_gen8_matches = results_df['reaches_gen8_match'].sum()
max_gen_matches = results_df['max_gen_match'].sum()
print(f"\nMatches between test3 and previous simulation:")
print(f"  Gen8 reach pattern matches: {reach_gen8_matches}/{len(results_df)}")
print(f"  Max generation matches: {max_gen_matches}/{len(results_df)}")

# Show mismatches
mismatches = results_df[~results_df['reaches_gen8_match']]
if len(mismatches) > 0:
    print(f"\nVariants with DIFFERENT gen8 reach pattern or missing previous data:")
    print(mismatches[['variant', 'gene', 'seed', 'test3_reaches_gen8', 'previous_reaches_gen8']].to_string(index=False))

# Show max gen mismatches
max_mismatches = results_df[~results_df['max_gen_match']]
if len(max_mismatches) > 0:
    print(f"\nVariants with DIFFERENT max generation or missing previous data:")
    print(max_mismatches[['variant', 'gene', 'seed', 'test3_max_gen', 'previous_max_gen']].to_string(index=False))

print(f"\nFirst 20 rows of results:")
print(results_df.head(20).to_string(index=False))


✓ Comparison results saved to: /user/home/il22158/work/vEcoli/surrogate/results/test3/test3_comparison_results.csv

Summary:
Total variants tested: 49
Rows (variants × seeds): 98

Matches between test3 and previous simulation:
  Gen8 reach pattern matches: 88/98
  Max generation matches: 88/98

Variants with DIFFERENT gen8 reach pattern or missing previous data:
 variant     gene  seed test3_reaches_gen8 previous_reaches_gen8
       0 BASELINE   100               True                  None
       0 BASELINE   101               True                  None
      14  EG10549   100              False                  True
      14  EG10549   101              False                  True
      28  EG10808   100               None                  True
      28  EG10808   101              False                  True
      34  EG10073   100              False                  True
      34  EG10073   101              False                  True
      39  EG10774   100               None        

## Test 4 - RNA/TU id run

In [5]:
import ast
import json
import pandas as pd
from pathlib import Path

base = Path('/user/home/il22158/work/vEcoli')
metadata_path = base / 'out/gene_knockout_TU_ID_test/variant_sim_data/metadata.json'
operon_csv_path = base / 'surrogate/operon_classification.csv'
gene_list_path = base / 'surrogate/RNA_TU_ID_test_gene_list.txt'

# 1) Load knockout targets from metadata (these may be TU IDs or RNA IDs).
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

gene_entries = []
knockout_map = metadata.get('gene_knockout', {})
for variant, payload in sorted(knockout_map.items(), key=lambda x: int(x[0])):
    for target_id in payload.get('genes_to_knockout', []):
        gene_entries.append((int(variant), str(target_id)))

# 2) Load operon classification and build lookup maps.
operon_df = pd.read_csv(operon_csv_path)

# TU -> list of gene IDs in that TU
tu_to_genes = {}
for tu_id, group in operon_df.groupby('tu_id', dropna=True):
    tu_to_genes[str(tu_id)] = sorted(set(group['id'].astype(str).tolist()))

# RNA ID -> list of gene IDs (using the rna_ids column, e.g. EG10016_RNA).
rna_to_genes = {}
for _, row in operon_df.iterrows():
    gene_id = str(row['id'])
    raw_rna_ids = row.get('rna_ids', '')
    if pd.isna(raw_rna_ids):
        continue
    try:
        parsed_rna_ids = ast.literal_eval(raw_rna_ids) if isinstance(raw_rna_ids, str) else []
    except Exception:
        parsed_rna_ids = []
    for rid in parsed_rna_ids:
        rid = str(rid).strip()
        if not rid:
            continue
        rna_to_genes.setdefault(rid, set()).add(gene_id)
        rna_to_genes.setdefault(f'{rid}[c]', set()).add(gene_id)

# 3) Convert knockout targets to gene IDs.
resolved_genes = []
resolution_log = []
unresolved_targets = []

for variant, target_id in gene_entries:
    mapped = []

    # Case A: target is a TU ID (e.g. TU0-1002[c]).
    if target_id in tu_to_genes:
        mapped = tu_to_genes[target_id]

    # Case B: target is an RNA ID (e.g. EG10016_RNA or EG10016_RNA[c]).
    elif target_id in rna_to_genes:
        mapped = sorted(rna_to_genes[target_id])

    # Case C: already a gene ID.
    elif target_id in set(operon_df['id'].astype(str)):
        mapped = [target_id]

    # Case D: try without compartment suffix.
    elif target_id.endswith('[c]') and target_id[:-3] in rna_to_genes:
        mapped = sorted(rna_to_genes[target_id[:-3]])

    if mapped:
        resolved_genes.extend(mapped)
        resolution_log.append((variant, target_id, mapped))
    else:
        unresolved_targets.append((variant, target_id))

# Keep unique genes in first-seen order
ordered_gene_ids = []
seen = set()
for gid in resolved_genes:
    if gid not in seen:
        seen.add(gid)
        ordered_gene_ids.append(gid)

# 4) Write converted gene list for gene_screen.py
with open(gene_list_path, 'w') as f:
    for gid in ordered_gene_ids:
        f.write(f'{gid}\n')

print(f'Created converted gene list: {gene_list_path}')
print(f'Gene count: {len(ordered_gene_ids)}')

print('\nResolved metadata targets -> gene IDs:')
for variant, target_id, mapped in resolution_log:
    print(f'  variant {variant}: {target_id} -> {mapped}')

if unresolved_targets:
    print('\nUnresolved targets (no mapping found in operon_classification.csv):')
    for variant, target_id in unresolved_targets:
        print(f'  variant {variant}: {target_id}')

cmd = (
    'python reading/gene_screen.py '
    '--project gene_knockout_TU_ID_test '
    '--lineage-seed 100 101 '
    '--variants 1 2 '
    f'--gene-list {gene_list_path}'
)

print('\nSuggested terminal command:')
print(cmd)

Created converted gene list: /user/home/il22158/work/vEcoli/surrogate/RNA_TU_ID_test_gene_list.txt
Gene count: 6

Resolved metadata targets -> gene IDs:
  variant 1: EG10016_RNA[c] -> ['EG10016']
  variant 2: TU0-1002[c] -> ['G6940', 'G6941', 'G6942', 'G6943', 'G6944']

Suggested terminal command:
python reading/gene_screen.py --project gene_knockout_TU_ID_test --lineage-seed 100 101 --variants 1 2 --gene-list /user/home/il22158/work/vEcoli/surrogate/RNA_TU_ID_test_gene_list.txt


**Understanding the gene_screen function and why a group of genes can have the same RNA acitvity and varied protein activity**:
- Same TU means same mRNA trace: the screen reads mRNA by TU from `rna_data` after the gene-to-RNA lookup in [reading/gene_expression_trace.py](reading/gene_expression_trace.py).
- Protein can differ because each gene is matched to its own cistron and then to its own monomer in `monomer_data`; one operon can share one TU but still encode different proteins.
- The gene→RNA mapping comes from [rnas.tsv](reconstruction/ecoli/flat/rnas.tsv), while the TU/gene grouping used for this test is in [operon_classification.csv](surrogate/operon_classification.csv).
- How operon_classification is constructed: taking use of *sim_data.process.transcription.cistron_id_to_rna_indexes* to covert gene ids to rna ids.
- So the observed pattern is expected: same mRNA within an operon, but different protein levels across the genes in that operon.

## Final run - Simluate the rest of 462 genes

**Gene_knockout function: Cistron lookup and TU conversion**

- The knockout code reads cistron metadata from `sim_data.process.transcription.cistron_data.struct_array`.
- For each gene ID, it matches `cistron["gene_id"]` to find the cistron record.
- It then converts the cistron to the corresponding TU/RNA index or indexes with `transcription.cistron_id_to_rna_indexes(cistron_id)`. (Original doc: [rnas.tsv](/user/home/il22158/work/vEcoli/reconstruction/ecoli/flat/rnas.tsv), coverted table: [operon_classification.tsv](/user/home/il22158/work/vEcoli/reading/results/knockout_experiment/operon_classification.tsv))
- The resolved TU indexes are deduplicated and passed to `sim_data.adjust_final_expression(..., [0.0, ...])` to apply the knockout.
- (**New function added after third test**) If the input already matches an RNA/TU ID, the updated `apply_variant()` can use `rna_data` directly instead of doing the cistron lookup.
- In short: gene ID input is converted to TU indexes through cistron data; TU ID input bypasses that lookup and goes straight to the RNA table.


In [7]:
import pandas as pd
from pathlib import Path

# Working directory
base = "/user/home/il22158/work/vEcoli/"

# Load previous simulation genes from CSV
csv_path = Path(f'{base}surrogate/results/ko_runs_with_success_generation.csv')
df = pd.read_csv(csv_path)

# Parse gene IDs from label column (format: "KO: EG10109")
previous_genes = set()
for label in df['label'].unique():
    if pd.notna(label) and label.startswith('KO: '):
        gene_id = label.replace('KO: ', '').strip()
        previous_genes.add(gene_id)

print(f"Previous simulation genes: {len(previous_genes)}")

# Load exclusion lists
third_round_path = Path(f'{base}surrogate/third_round_tested_gene_list.txt')
with open(third_round_path) as f:
    third_round_genes = set(line.strip() for line in f if line.strip())

outlier_path = Path(f'{base}surrogate/results/failure/outlier_genes_all_unique.txt')
with open(outlier_path) as f:
    outlier_genes = set(line.strip() for line in f if line.strip())

exclusion_genes = third_round_genes | outlier_genes
print(f"3rd round genes: {len(third_round_genes)}")
print(f"Outlier/p_list genes: {len(outlier_genes)}")
print(f"Total exclusions: {len(exclusion_genes)}")

# Calculate new genes (not yet tested)
new_genes = previous_genes - exclusion_genes
print(f"New genes to test (excluding duplicates): {len(new_genes)}")

# Save new gene list to csv file
new_genes_df = pd.DataFrame({'gene_id': list(new_genes)})
new_genes_csv_path = Path(f'{base}surrogate/rest_genes_to_test.csv')
new_genes_df.to_csv(new_genes_csv_path, index=False)
print(f"New gene list saved to: {new_genes_csv_path}")

Previous simulation genes: 462
3rd round genes: 50
Outlier/p_list genes: 52
Total exclusions: 102
New genes to test (excluding duplicates): 360
New gene list saved to: /user/home/il22158/work/vEcoli/surrogate/rest_genes_to_test.csv


In [6]:
# Check genes that failed before gen8 and are NOT in p_list
failed_before_gen8 = df[df['failed_before_gen8'] == 1]['gene_id'].unique()
print(f"Total genes that failed before gen8: {len(failed_before_gen8)}")

# Genes that failed before gen8 but are NOT in the outlier/p_list set
failed_not_in_plist = set(failed_before_gen8) - outlier_genes
print(f"Failed before gen8 but NOT in p_list: {len(failed_not_in_plist)}")
print(f"Genes: {sorted(failed_not_in_plist)}")

# Genes that failed before gen8 AND ARE in p_list
failed_in_plist = set(failed_before_gen8) & outlier_genes
print(f"\nFailed before gen8 AND in p_list: {len(failed_in_plist)}")
print(f"Genes: {sorted(failed_in_plist)}")

# Saved the failed but not in p_list genes to a separate CSV for review
failed_not_in_plist_df = pd.DataFrame({'gene_id': sorted(failed_not_in_plist)})
failed_not_in_plist_csv_path = Path(f'{base}surrogate/failed_before_gen8_not_in_plist.csv')
failed_not_in_plist_df.to_csv(failed_not_in_plist_csv_path, index=False)
print(f"Failed before gen8 but not in p_list genes saved to: {failed_not_in_plist_csv_path}")

Total genes that failed before gen8: 117
Failed before gen8 but NOT in p_list: 65
Genes: ['EG10034', 'EG10063', 'EG10071', 'EG10074', 'EG10078', 'EG10081', 'EG10094', 'EG10097', 'EG10187', 'EG10205', 'EG10208', 'EG10383', 'EG10390', 'EG10407', 'EG10409', 'EG10410', 'EG10444', 'EG10445', 'EG10446', 'EG10447', 'EG10448', 'EG10449', 'EG10450', 'EG10451', 'EG10453', 'EG10492', 'EG10496', 'EG10497', 'EG10532', 'EG10581', 'EG10586', 'EG10709', 'EG10710', 'EG10769', 'EG10770', 'EG10793', 'EG10794', 'EG10796', 'EG10797', 'EG10807', 'EG10810', 'EG10871', 'EG10873', 'EG10878', 'EG10893', 'EG10894', 'EG10895', 'EG10903', 'EG10910', 'EG10912', 'EG10947', 'EG10999', 'EG11000', 'EG11001', 'EG11027', 'EG11028', 'EG11030', 'EG11039', 'EG11043', 'EG11067', 'EG11226', 'EG11575', 'EG11576', 'EG11577', 'G6879']

Failed before gen8 AND in p_list: 52
Genes: ['EG10196', 'EG10707', 'EG10864', 'EG10865', 'EG10866', 'EG10867', 'EG10868', 'EG10869', 'EG10870', 'EG10872', 'EG10874', 'EG10875', 'EG10876', 'EG10877

In [9]:
"""Covert gene ids to TU ids to avoid duplicating tests of genes in the same TU in the rest gene list"""
# Base path
base = "/user/home/il22158/work/vEcoli/"

# Load operon classification and target gene list
operon_path = Path(f'{base}reading/results/knockout_experiment/operon_classification.tsv')
operon_df = pd.read_csv(operon_path, sep='\t')
target_genes_path = Path(f'{base}surrogate/rest_genes_to_test.csv')
target_genes_df = pd.read_csv(target_genes_path)

# Create mapping from gene ID to TU ID
gene_to_tu = dict(zip(operon_df['id'], operon_df['tu_id']))

# Map new genes to TU IDs and deduplicate
tu_ids = set()
genes_found = []
genes_not_found = []

for gene_id in sorted(new_genes):
    if gene_id in gene_to_tu:
        tu_id = gene_to_tu[gene_id]
        tu_ids.add(tu_id)
        genes_found.append(gene_id)
    else:
        genes_not_found.append(gene_id)

print(f"\nGenes found in operon classification: {len(genes_found)}")
print(f"Genes not found: {len(genes_not_found)}")
if genes_not_found:
    print(f"  Not found: {genes_not_found[:5]}..." if len(genes_not_found) > 5 else f"  Not found: {genes_not_found}")

print(f"\nUnique TU IDs to test (operon-level): {len(tu_ids)}")
print(f"Unique TU IDs (sample): {sorted(tu_ids)[:5]}")

# Display the TU list
print(f"\nAll TU IDs for new experiments:")
for tu_id in sorted(tu_ids):
    print(tu_id)
    
# Save TU list to csv file
tu_ids_df = pd.DataFrame({'tu_id': sorted(tu_ids)})
tu_ids_csv_path = Path(f'{base}surrogate/rest_tu_ids_to_test.csv')
tu_ids_df.to_csv(tu_ids_csv_path, index=False)
print(f"TU ID list saved to: {tu_ids_csv_path}")


Genes found in operon classification: 360
Genes not found: 0

Unique TU IDs to test (operon-level): 292
Unique TU IDs (sample): ['EG10016_RNA[c]', 'EG10214_RNA[c]', 'EG10383_RNA[c]', 'EG10982_RNA[c]', 'EG12662_RNA[c]']

All TU IDs for new experiments:
EG10016_RNA[c]
EG10214_RNA[c]
EG10383_RNA[c]
EG10982_RNA[c]
EG12662_RNA[c]
EG12663_RNA[c]
TU0-1002[c]
TU0-1021[c]
TU0-1063[c]
TU0-12797[c]
TU0-12810[c]
TU0-12812[c]
TU0-12827[c]
TU0-12831[c]
TU0-12833[c]
TU0-12921[c]
TU0-12924[c]
TU0-12942[c]
TU0-12947[c]
TU0-12962[c]
TU0-12963[c]
TU0-13010[c]
TU0-13018[c]
TU0-13066[c]
TU0-13069[c]
TU0-13078[c]
TU0-13080[c]
TU0-13084[c]
TU0-13087[c]
TU0-13093[c]
TU0-13100[c]
TU0-13106[c]
TU0-13135[c]
TU0-13182[c]
TU0-13189[c]
TU0-13209[c]
TU0-13215[c]
TU0-13350[c]
TU0-13373[c]
TU0-13440[c]
TU0-13444[c]
TU0-13499[c]
TU0-13514[c]
TU0-13519[c]
TU0-13539[c]
TU0-13552[c]
TU0-13565[c]
TU0-13580[c]
TU0-13593[c]
TU0-13627[c]
TU0-13637[c]
TU0-13642[c]
TU0-13651[c]
TU0-13713[c]
TU0-13736[c]
TU0-13741[c]
TU0-13783[

In [10]:
# Covert the mapping table from tsv to csv
original_file = "/user/home/il22158/work/vEcoli/reading/results/knockout_experiment/operon_classification.tsv"
operon_df = pd.read_csv(original_file, sep='\t')
csv_output_path = "/user/home/il22158/work/vEcoli/surrogate/operon_classification.csv"
operon_df.to_csv(csv_output_path, index=False)
print(f"Operon classification table saved to: {csv_output_path}")

Operon classification table saved to: /user/home/il22158/work/vEcoli/surrogate/operon_classification.csv


In [11]:
from ast import literal_eval

# Check whether the remaining gene list covers all genes in each multi-gene operon.
# This helps verify that moving from gene-level to operon-level does not leave out
# any genes that share the same TU.

def parse_gene_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = literal_eval(text)
            if isinstance(parsed, list):
                return [item.strip() for item in parsed if isinstance(item, str) and item.strip()]
        except Exception:
            pass
        return [item.strip() for item in text.split(',') if item.strip()]
    return []

# Build the remaining gene set from the notebook table, if needed.
if 'target_genes_df' in globals() and not target_genes_df.empty:
    if 'gene_id' in target_genes_df.columns:
        rest_gene_to_test = set(target_genes_df['gene_id'].astype(str).str.strip())
    elif 'id' in target_genes_df.columns:
        rest_gene_to_test = set(target_genes_df['id'].astype(str).str.strip())
    else:
        rest_gene_to_test = set(target_genes_df.iloc[:, 0].astype(str).str.strip())
else:
    rest_gene_to_test = set(new_genes)

print(f"Remaining genes to test: {len(rest_gene_to_test)}")

# Focus on operons represented in the remaining gene list.
rest_operons = operon_df[operon_df['id'].isin(rest_gene_to_test)].copy()
rest_operons['operon_genes'] = rest_operons.apply(
    lambda row: sorted(set([row['id']] + parse_gene_list(row['other_genes_in_operon']))),
    axis=1,
)

multi_gene_operons = rest_operons[rest_operons['operon_size'].fillna(1).astype(float) > 1].copy()
print(f"Genes in multi-gene operons: {len(multi_gene_operons)}")
print(f"Unique multi-gene operons touched by rest_gene_to_test: {multi_gene_operons['tu_id'].nunique()}")

# Check whether the list covers every gene in each operon.
operon_gaps = []
for tu_id, group in multi_gene_operons.groupby('tu_id'):
    operon_genes = set()
    for _, row in group.iterrows():
        operon_genes.update(row['operon_genes'])
    present_genes = operon_genes & rest_gene_to_test
    missing_genes = sorted(operon_genes - rest_gene_to_test)
    if missing_genes:
        operon_gaps.append({
            'tu_id': tu_id,
            'operon_genes': sorted(operon_genes),
            'present_genes': sorted(present_genes),
            'missing_genes': missing_genes,
        })

print(f"Operons where rest_gene_to_test does NOT cover all genes: {len(operon_gaps)}")

if operon_gaps:
    print("\nExamples of incomplete operon coverage:")
    for gap in operon_gaps[:10]:
        print(f"- {gap['tu_id']}: present={gap['present_genes']}, missing={gap['missing_genes']}")
else:
    print("\nAll multi-gene operons touched by rest_gene_to_test are fully covered.")

# Optional: summarize the operons that are fully covered.
covered_operons = []
for tu_id, group in multi_gene_operons.groupby('tu_id'):
    operon_genes = set()
    for _, row in group.iterrows():
        operon_genes.update(row['operon_genes'])
    if operon_genes.issubset(rest_gene_to_test):
        covered_operons.append((tu_id, sorted(operon_genes)))

print(f"\nFully covered multi-gene operons: {len(covered_operons)}")
for tu_id, genes in covered_operons[:10]:
    print(f"- {tu_id}: {genes}")


Remaining genes to test: 360
Genes in multi-gene operons: 225
Unique multi-gene operons touched by rest_gene_to_test: 157
Operons where rest_gene_to_test does NOT cover all genes: 139

Examples of incomplete operon coverage:
- TU0-1002[c]: present=['G6941', 'G6943'], missing=['G6940', 'G6942', 'G6944']
- TU0-12810[c]: present=['EG12312'], missing=['EG12313', 'EG12314']
- TU0-12827[c]: present=['EG10207'], missing=['EG10570', 'EG11411']
- TU0-12921[c]: present=['G6234'], missing=['EG10666', 'EG10704', 'EG11320', 'EG11321', 'EG11322']
- TU0-12924[c]: present=['G6239'], missing=['G198']
- TU0-12962[c]: present=['EG12384'], missing=['G6287', 'G6288', 'G6289']
- TU0-12963[c]: present=['EG12666'], missing=['EG10758']
- TU0-13010[c]: present=['EG10532'], missing=['EG10855', 'EG11412', 'G6349', 'G6350']
- TU0-13069[c]: present=['G6442'], missing=['G6443']
- TU0-13087[c]: present=['EG10613'], missing=['EG11409', 'EG12375', 'G6471']

Fully covered multi-gene operons: 18
- TU0-12831[c]: ['EG10139

In [14]:
# Build a TU-level output table that keeps track of which remaining genes map to each TU.
# Also flag whether the operon contains genes that are not present in rest_gene_to_test.

tu_to_rest_genes = {}
tu_to_operon_genes = {}
for _, row in operon_df.iterrows():
    gene_id = row['id']
    tu_id = row['tu_id']
    operon_genes = sorted(set([gene_id] + parse_gene_list(row['other_genes_in_operon'])))
    tu_to_operon_genes[tu_id] = operon_genes
    if gene_id in rest_gene_to_test:
        tu_to_rest_genes.setdefault(tu_id, []).append(gene_id)

# Prefer the TU IDs we already derived earlier, but fall back to the keys from the mapping.
final_tu_ids = sorted(tu_ids) if 'tu_ids' in globals() and tu_ids else sorted(tu_to_rest_genes)

# Create the CSV with extra columns showing the genes in rest_gene_to_test that map to each TU
# and whether the operon includes additional genes outside the remaining gene list.
tu_ids_df = pd.DataFrame(
    {
        'tu_id': final_tu_ids,
        'genes_in_rest_gene_to_test': [', '.join(tu_to_rest_genes.get(tu_id, [])) for tu_id in final_tu_ids],
        'n_genes_in_rest_gene_to_test': [len(tu_to_rest_genes.get(tu_id, [])) for tu_id in final_tu_ids],
        'operon_has_additional_genes_not_in_rest_gene_to_test': [
            len(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test) > 0
            for tu_id in final_tu_ids
        ],
        'n_additional_genes_not_in_rest_gene_to_test': [
            len(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test)
            for tu_id in final_tu_ids
        ],
        'genes_not_in_rest_gene_to_test': [
            ', '.join(sorted(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test))
            for tu_id in final_tu_ids
        ],
    }
)

tu_ids_csv_path = Path(f'{base}surrogate/rest_tu_ids_to_test.csv')
tu_ids_df.to_csv(tu_ids_csv_path, index=False)

print(f"TU ID table saved to: {tu_ids_csv_path}")
print(f"Rows written: {len(tu_ids_df)}")
print(tu_ids_df.head(10).to_string(index=False))


TU ID table saved to: /user/home/il22158/work/vEcoli/surrogate/rest_tu_ids_to_test.csv
Rows written: 292
         tu_id genes_in_rest_gene_to_test  n_genes_in_rest_gene_to_test  operon_has_additional_genes_not_in_rest_gene_to_test  n_additional_genes_not_in_rest_gene_to_test genes_not_in_rest_gene_to_test
EG10016_RNA[c]                    EG10016                             1                                                 False                                            0                               
EG10214_RNA[c]                    EG10214                             1                                                 False                                            0                               
EG10383_RNA[c]                    EG10383                             1                                                 False                                            0                               
EG10982_RNA[c]                    EG10982                             1                

## Debug 2
How to start new simulation following the early-stopped previous simulation?

**Solution: Restart from a previous run's last state**

- Summary: the simulator can load saved daughter_state JSONs as `--initial_state_file`; to start a new experiment from a previous run's final generation you must supply, per variant+seed, the matching state file into the first simulation job. Modifying the workflow to pass these files automatically is the recommended approach.

**Steps to follow**

1. Locate the final state JSONs from the old run:
   - `out/<OLD_EXPERIMENT>/nextflow/daughter_states/variant=<V>/lineage_seed=<S>/generation=<G>/daughter_state_*.json`.
2. Recommended — patch the workflow (minimal, repeatable):
   - Add a per-variant+seed manifest (or use `initial_state_overrides` in the config) mapping `(variant, seed) -> initial_state_path` and optional `initial_global_time`.
   - Update `runscripts/workflow.py` to write that mapping into the generated `workflow_config.json`.
   - Update `runscripts/nextflow/template.nf` so the `variantCh` includes `initial_state` (and `initial_global_time`) and pass them to `simGen0` (the first sim process).
   - Run the workflow with the updated config: `python3 runscripts/workflow.py --config <your_config.json>`.
3. Quick workaround (manual):
   - Start an interactive container or SLURM job and run `ecoli_master_sim.py` per variant/seed with `--initial_state_file <path>` and `--initial_global_time <time>`.
4. Use Sherlock/HyperQueue best practices for throughput:
   - Set `HYPERQUEUE: true`, reuse a built `container_image` (`sherlock.build_image=false`), set `emitter_arg.out_dir` to `$SCRATCH`, set `emitter_arg.threaded=false`, and tune `HQ_CORES` / `SIM_CPUS` / `SIM_MEM` to match cluster limits.
5. Validate: test one variant+seed first to confirm the initial state is loaded and history matches expectations, then run the full experiment.

If you want, I can (A) generate the patch for `runscripts/workflow.py` + `nextflow/template.nf` that injects the per-variant initial_state mapping, or (B) create a small script that builds `initial_state_overrides` from the old output. Which do you prefer?

**Goal**
Run a new workflow that uses the previous run's final daughter_state JSONs as initial states for each `(variant, lineage_seed)` — without modifying `ecoli_master_sim.py`.

**Quick steps (minimal changes)**

1. Collect final daughter_state JSONs from the old experiment (absolute paths):

```bash
find /full/path/to/out/<OLD_EXPERIMENT>/nextflow -type f -name 'daughter_state_*.json' > /tmp/daughter_states.txt
```

2. Build `initial_state_overrides` JSON listing mappings of `variant`, `lineage_seed`, `initial_state_file`, and `initial_global_time` (example):

```json
{
  "initial_state_overrides": [
    {"variant": 1, "lineage_seed": 100, "initial_state_file": "/full/path/to/daughter_state_variant1_seed100.json", "initial_global_time": 24.0},
    {"variant": 2, "lineage_seed": 100, "initial_state_file": "/full/path/to/daughter_state_variant2_seed100.json", "initial_global_time": 24.0}
  ]
}
```

Save this file (for example) as `configs/initial_states.json` and merge it with your normal config or pass it as your config.

3. Launch the workflow (use a new `experiment_id` or set `--resume` as needed):

```bash
python3 runscripts/workflow.py --config configs/initial_states.json
```

4. Sherlock/HyperQueue notes (if applicable):
- Set `HYPERQUEUE: true` and tune `HQ_CORES` / `SIM_CPUS` / `SIM_MEM` for throughput.
- Reuse a built Apptainer image (`sherlock.build_image = false` and `sherlock.container_image = <path>`) to avoid rebuilds.
- Put `emitter_arg.out_dir` on `$SCRATCH` and set `emitter_arg.threaded = false` for Parquet emitter on Sherlock.

5. Optional automation: write a tiny script that reads `/tmp/daughter_states.txt`, extracts `variant` and `lineage_seed` from the file paths, and emits `initial_state_overrides` JSON.

**Reminder**: Do not change `ecoli_master_sim.py` — it already supports `--initial_state_file` and `--initial_global_time`.

The **slurm log** shows the workflow reached sim_gen_4 and then was cancelled for time limit, with 580 of 580 in gen1, 560 of 560 in gen2, and 542 of 545 in gen3 before only 1 of 496 in gen4.
```
executor >  local (1708)
[1c/aa1298] createVariants                 | 1 of 1 ✔
[4e/8cf1cf] sim…0/generation=1/agent_id=0) | 580 of 580, ignored: 20 ✔
[eb/d21240] sim…/generation=2/agent_id=00) | 560 of 560, ignored: 15 ✔
[7e/36e4bb] sim…generation=3/agent_id=000) | 542 of 545, ignored: 46
[49/a86f28] sim…eneration=4/agent_id=0000) | 1 of 496
[-        ] sim_gen_5                      | 0 of 1
[-        ] sim_gen_6                      -
[-        ] sim_gen_7                      -
[-        ] sim_gen_8                      -
slurmstepd: error: *** JOB 17298699 ON bp1-compute218 CANCELLED AT 2026-05-15T15:37:46 DUE TO TIME LIMIT ***```

So for different variants we want to start with different generation number - and the goal is to make the sum of new simulation and previous simulation total generation numnber to be 8.

Therefore we want to create separate cofiguration file for these grouped variants.

In [2]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

base = Path('/user/home/il22158/work/vEcoli')
raw_root = base / 'out' / 'gene_knockout_TU_ID_rest_list'

rows = []
variant_seed_max = defaultdict(dict)

# Prefer success/experiment_id layout if present
success_root = raw_root / 'success'
if success_root.exists():
    for exp_dir in success_root.glob('experiment_id=*'):
        for variant_dir in exp_dir.glob('variant=*'):
            if not variant_dir.is_dir():
                continue
            variant = int(variant_dir.name.split('=')[1])
            for seed_dir in variant_dir.glob('lineage_seed=*'):
                try:
                    seed = int(seed_dir.name.split('=')[1])
                except Exception:
                    continue
                generation_dirs = [p for p in seed_dir.glob('generation=*') if p.is_dir()]
                if not generation_dirs:
                    continue
                max_generation = max(int(p.name.split('=')[1]) for p in generation_dirs)
                rows.append((variant, seed, max_generation))
                variant_seed_max[variant][seed] = max_generation
else:
    # fallback to daughter_states layout
    daughter_root = raw_root / 'daughter_states'
    for variant_dir in daughter_root.glob('variant=*'):
        if not variant_dir.is_dir():
            continue
        variant = int(variant_dir.name.split('=')[1])
        for seed_dir in list(variant_dir.glob('seed=*')) + list(variant_dir.glob('lineage_seed=*')):
            try:
                seed = int(seed_dir.name.split('=')[1])
            except Exception:
                continue
            generation_dirs = [p for p in seed_dir.glob('generation=*') if p.is_dir()]
            if not generation_dirs:
                continue
            max_generation = max(int(p.name.split('=')[1]) for p in generation_dirs)
            rows.append((variant, seed, max_generation))
            variant_seed_max[variant][seed] = max_generation

rows_df = pd.DataFrame(rows, columns=['variant', 'lineage_seed', 'max_success_generation'])
all_variants = sorted(rows_df['variant'].unique().tolist())

print(f'Raw variant/seed rows: {len(rows_df)}')
print(f'Unique variants present: {len(all_variants)}\n')

# For generations 1..4, print variants that have any seed >= gen and variants where all recorded seeds >= gen
for gen in range(1, 5):
    variants_any = sorted({v for v, s, g in rows if g >= gen})
    variants_all = []
    for v in sorted(variant_seed_max.keys()):
        seeds = variant_seed_max[v]
        if seeds and all(mg >= gen for mg in seeds.values()):
            variants_all.append(v)
    print(f'Generation {gen}:')
    print(f'  Variants with at least one seed reaching >= {gen}: {len(variants_any)}')
    if variants_any:
        print(f'    Sample: {variants_any[:50]}')
    print(f'  Variants where ALL recorded seeds reached >= {gen}: {len(variants_all)}')
    if variants_all:
        print(f'    Sample: {variants_all[:50]}')
    print('')

# Additionally show counts of variants grouped by their recorded max-generation (for quick overview)
group_counts = rows_df.groupby('max_success_generation')['variant'].nunique().sort_index()
print('Variants present per max-success-generation (unique variant count):')
for gen, cnt in group_counts.items():
    print(f'  gen {int(gen)}: {int(cnt)} variants')

print('\nNote: "Variants with at least one seed reaching >= G" counts variants that have any seed with max_success_generation >= G.')

Raw variant/seed rows: 553
Unique variants present: 277

Generation 1:
  Variants with at least one seed reaching >= 1: 277
    Sample: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
  Variants where ALL recorded seeds reached >= 1: 277
    Sample: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

Generation 2:
  Variants with at least one seed reaching >= 2: 272
    Sample: [1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]
  Variants where ALL recorded seeds reached >= 2: 243
    Sample: [1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22

In [6]:
# Compute per-variant maximum generation (across seeds) and show counts
if 'rows_df' not in globals():
    raise RuntimeError('rows_df not found; run the previous cell first')
per_variant_max = rows_df.groupby('variant')['max_success_generation'].max()
counts = per_variant_max.value_counts().sort_index()
print('Variants present per per-variant max-success-generation (each variant counted once):')
for gen, cnt in counts.items():
    print(f'  gen {int(gen)}: {int(cnt)} variants')

# Sanity checks: how many variant+seed rows have each max_generation
row_counts = rows_df['max_success_generation'].value_counts().sort_index()
print('Per-seed rows per max_success_generation (total rows):')
for gen, cnt in row_counts.items():
    print(f'  gen {int(gen)}: {int(cnt)} rows (variant+seed)')

print('')

Variants present per per-variant max-success-generation (each variant counted once):
  gen 1: 5 variants
  gen 2: 33 variants
  gen 3: 238 variants
  gen 4: 1 variants
Per-seed rows per max_success_generation (total rows):
  gen 1: 38 rows (variant+seed)
  gen 2: 38 rows (variant+seed)
  gen 3: 476 rows (variant+seed)
  gen 4: 1 rows (variant+seed)



In [4]:
import json
from pathlib import Path
from collections import defaultdict

# This cell uses `rows_df` produced by the previous cell
if 'rows_df' not in globals():
    raise RuntimeError('rows_df not found; run the previous cell first')

base = Path('/user/home/il22158/work/vEcoli')
raw_root = base / 'out' / 'gene_knockout_TU_ID_rest_list'
configs_dir = base / 'configs'
configs_dir.mkdir(parents=True, exist_ok=True)

target_total = 8
overrides_by_add = defaultdict(list)

# rows_df has columns: variant, lineage_seed, max_success_generation
for _, row in rows_df.iterrows():
    variant = int(row['variant'])
    seed = int(row['lineage_seed'])
    g = int(row['max_success_generation'])
    if g >= target_total:
        continue
    add = target_total - g
    start_generation = g + 1
    candidate = None

    # Try success/experiment layout first (search recursively under generation dirs)
    success_root = raw_root / 'success'
    if success_root.exists():
        for exp_dir in success_root.glob('experiment_id=*'):
            gen_dir = exp_dir / f'variant={variant}' / f'lineage_seed={seed}' / f'generation={g}'
            if gen_dir.exists():
                files = list(gen_dir.glob('**/daughter_state_*.json'))
                if files:
                    candidate = files[0].resolve()
                    break
    # Try daughter_states layout
    if candidate is None:
        daughter_root = raw_root / 'daughter_states'
        variant_dir = daughter_root / f'variant={variant}'
        for sd in [variant_dir / f'seed={seed}', variant_dir / f'lineage_seed={seed}']:
            gen_dir = sd / f'generation={g}'
            if gen_dir.exists():
                files = list(gen_dir.glob('**/daughter_state_*.json'))
                if files:
                    candidate = files[0].resolve()
                    break
    # Broader search as a last resort (allow any nesting under generation)
    if candidate is None:
        files = list(raw_root.glob(f'**/variant={variant}/**/generation={g}/**/daughter_state_*.json'))
        if files:
            candidate = files[0].resolve()

    if candidate is None:
        print(f'Warning: no daughter_state found for variant={variant} seed={seed} gen={g}')
        continue

    # Heuristic for initial_global_time; adjust if you have a more accurate mapping
    initial_global_time = float(g) * 24.0
    overrides_by_add[add].append({
        'variant': int(variant),
        'lineage_seed': int(seed),
        'initial_state_file': str(candidate),
        'initial_global_time': initial_global_time,
        'start_generation': int(start_generation),
        'resume_generations': int(add)
    })

# Write one config per grouped 'additional generations' value
for add, lst in sorted(overrides_by_add.items()):
    fname = configs_dir / f'resume_add_{add}_gens_to_8.json'
    payload = { 'target_total_generation': target_total, 'initial_state_overrides': lst }
    with open(fname, 'w') as fh:
        json.dump(payload, fh, indent=2)
    print(f'Wrote {len(lst)} overrides to {fname}')

print('Done. Review configs in', configs_dir)

Wrote 1 overrides to /user/home/il22158/work/vEcoli/configs/resume_add_4_gens_to_8.json
Wrote 476 overrides to /user/home/il22158/work/vEcoli/configs/resume_add_5_gens_to_8.json
Wrote 38 overrides to /user/home/il22158/work/vEcoli/configs/resume_add_6_gens_to_8.json
Wrote 38 overrides to /user/home/il22158/work/vEcoli/configs/resume_add_7_gens_to_8.json
Done. Review configs in /user/home/il22158/work/vEcoli/configs


In [9]:
import json
from pathlib import Path
import re
import copy

repo = Path('/user/home/il22158/work/vEcoli')
cfg_dir = repo / 'configs'
template_path = cfg_dir / 'N_gene_knockout_TU_ID_4th_gen.json'
if not template_path.exists():
    raise FileNotFoundError(f'Template not found: {template_path}')
with open(template_path, 'r') as fh:
    template = json.load(fh)

# Find resume_add files and create full configs from template
for f in sorted(cfg_dir.glob('resume_add_*_gens_to_8.json')):
    m = re.search(r'resume_add_(\d+)_gens_to_8.json$', f.name)
    if not m:
        continue
    add = int(m.group(1))
    with open(f, 'r') as fh:
        payload = json.load(fh)
    overrides = payload.get('initial_state_overrides', [])
    newcfg = copy.deepcopy(template)
    newcfg['experiment_id'] = f'N_gene_knockout_TU_ID_resume_add_{add}_to8'
    # Run only the additional generations when resuming from initial states
    newcfg['generations'] = int(add)
    newcfg['initial_state_overrides'] = overrides
    # Ensure emitter out_dir is absolute and points to repo/out by default
    if 'emitter_arg' in newcfg and 'out_dir' in newcfg['emitter_arg']:
        out_dir = newcfg['emitter_arg']['out_dir']
        if not Path(out_dir).is_absolute():
            newcfg['emitter_arg']['out_dir'] = str(repo / out_dir)
    out_path = cfg_dir / f'N_gene_knockout_TU_ID_resume_add_{add}_to8.json'
    with open(out_path, 'w') as fh:
        json.dump(newcfg, fh, indent=2)
    print(f'Wrote full config: {out_path} (overrides: {len(overrides)})')

print('Done: created resume configs in', cfg_dir)

Wrote full config: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_4_to8.json (overrides: 1)
Wrote full config: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_5_to8.json (overrides: 476)
Wrote full config: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_6_to8.json (overrides: 38)
Wrote full config: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_7_to8.json (overrides: 38)
Done: created resume configs in /user/home/il22158/work/vEcoli/configs


In [14]:
import json
import pandas as pd
from pathlib import Path
import re
import copy

# This cell filters the full template to include ONLY the variants in each resume config
# AND remaps variant numbers to match the filtered list (so variant numbers are sequential)
repo = Path('/user/home/il22158/work/vEcoli')
cfg_dir = repo / 'configs'
template_path = cfg_dir / 'N_gene_knockout_TU_ID_4th_gen.json'

if not template_path.exists():
    raise FileNotFoundError(f'Template not found: {template_path}')

with open(template_path, 'r') as fh:
    template = json.load(fh)

# Get all variant definitions from template (these are indexed by position starting at 1)
all_genes_to_knockout = template.get('variants', {}).get('gene_knockout', {}).get('genes_to_knockout', {}).get('value', [])
print(f'Template has {len(all_genes_to_knockout)} variant definitions\n')

# Store all mappings in a list for CSV output
all_mappings = []

# Process each resume_add_*_gens_to_8.json file and create filtered configs
for resume_add_path in sorted(cfg_dir.glob('resume_add_*_gens_to_8.json')):
    m = re.search(r'resume_add_(\d+)_gens_to_8.json$', resume_add_path.name)
    if not m:
        continue
    
    add = int(m.group(1))
    
    with open(resume_add_path, 'r') as fh:
        payload = json.load(fh)
    
    overrides = payload.get('initial_state_overrides', [])
    
    # Extract original variant numbers from overrides
    original_variant_numbers = sorted(set(o.get('variant') for o in overrides))
    print(f'resume_add_{add}_gens_to_8: {len(overrides)} overrides covering {len(original_variant_numbers)} unique variants')
    
    # Create new config from template
    newcfg = copy.deepcopy(template)
    newcfg['experiment_id'] = f'N_gene_knockout_TU_ID_resume_add_{add}_to8'
    newcfg['generations'] = int(add)
    
    # Filter gene_knockout to only include variants in the overrides
    # Variants are 1-indexed in the template (variant 1 is at index 0)
    filtered_genes = []
    old_to_new_variant_map = {}  # Maps original variant number to new variant number
    
    for new_variant_num, old_variant_num in enumerate(original_variant_numbers, start=1):
        if 1 <= old_variant_num <= len(all_genes_to_knockout):
            filtered_genes.append(all_genes_to_knockout[old_variant_num - 1])
            old_to_new_variant_map[old_variant_num] = new_variant_num
        else:
            print(f'  Warning: variant {old_variant_num} out of range (template has 1-{len(all_genes_to_knockout)})')
    
    # Update initial_state_overrides with NEW variant numbers
    new_overrides = []
    for override in overrides:
        old_variant = override.get('variant')
        new_variant = old_to_new_variant_map.get(old_variant)
        
        if new_variant is not None:
            new_override = copy.deepcopy(override)
            new_override['variant'] = new_variant
            new_overrides.append(new_override)
            
            # Record mapping for CSV
            all_mappings.append({
                'config_group': f'resume_add_{add}_to8',
                'original_variant': old_variant,
                'new_variant': new_variant,
                'lineage_seed': override.get('lineage_seed'),
                'resume_generations': override.get('resume_generations'),
                'initial_state_file': override.get('initial_state_file')
            })
        else:
            print(f'  Warning: variant {old_variant} not in mapping')
    
    # Update the config with filtered variants and remapped overrides
    newcfg['variants']['gene_knockout']['genes_to_knockout']['value'] = filtered_genes
    newcfg['initial_state_overrides'] = new_overrides
    
    # Ensure emitter out_dir is absolute
    if 'emitter_arg' in newcfg and 'out_dir' in newcfg['emitter_arg']:
        out_dir = newcfg['emitter_arg']['out_dir']
        if not Path(out_dir).is_absolute():
            newcfg['emitter_arg']['out_dir'] = str(repo / out_dir)
    
    # Write filtered config
    out_path = cfg_dir / f'N_gene_knockout_TU_ID_resume_add_{add}_to8.json'
    with open(out_path, 'w') as fh:
        json.dump(newcfg, fh, indent=2)
    
    print(f'  ✓ Wrote config with {len(filtered_genes)} variants: {out_path}')
    print(f'    Original variants {len(original_variant_numbers)} → New variants 1-{len(original_variant_numbers)}')

# Save variant mapping to CSV
if all_mappings:
    mapping_df = pd.DataFrame(all_mappings)
    mapping_csv_path = cfg_dir / 'resume_configs_variant_mapping.csv'
    mapping_df.to_csv(mapping_csv_path, index=False)
    print(f'\n✓ Variant mapping saved to: {mapping_csv_path}')
    print(f'  Total override entries: {len(mapping_df)}')
    print(f'\nFirst 10 rows:')
    print(mapping_df.head(10).to_string(index=False))

print('\nDone: created filtered resume configs with remapped variants')


Template has 290 variant definitions

resume_add_4_gens_to_8: 1 overrides covering 1 unique variants
  ✓ Wrote config with 1 variants: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_4_to8.json
    Original variants 1 → New variants 1-1
resume_add_5_gens_to_8: 476 overrides covering 239 unique variants
  ✓ Wrote config with 239 variants: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_5_to8.json
    Original variants 239 → New variants 1-239
resume_add_6_gens_to_8: 38 overrides covering 34 unique variants
  ✓ Wrote config with 34 variants: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_6_to8.json
    Original variants 34 → New variants 1-34
resume_add_7_gens_to_8: 38 overrides covering 34 unique variants
  ✓ Wrote config with 34 variants: /user/home/il22158/work/vEcoli/configs/N_gene_knockout_TU_ID_resume_add_7_to8.json
    Original variants 34 → New variants 1-34

✓ Variant mapping saved to: /user/home/il22158/wo

In [16]:
import json
import pandas as pd
import re
from pathlib import Path

# Load the mapping CSV to understand the remapping
repo = Path('/user/home/il22158/work/vEcoli')
cfg_dir = repo / 'configs'
mapping_csv_path = cfg_dir / 'resume_configs_variant_mapping.csv'

if not mapping_csv_path.exists():
    print(f"ERROR: Mapping CSV not found: {mapping_csv_path}")
else:
    mapping_df = pd.read_csv(mapping_csv_path)
    print(f"Loaded mapping CSV: {len(mapping_df)} rows\n")
    
    # For each config, validate and fix if needed
    for config_file in sorted(cfg_dir.glob('N_gene_knockout_TU_ID_resume_add_*_to8.json')):
        print(f"\n{'='*70}")
        print(f"Validating: {config_file.name}")
        print(f"{'='*70}")
        
        with open(config_file, 'r') as fh:
            config = json.load(fh)
        
        overrides = config.get('initial_state_overrides', [])
        config_group = config['experiment_id']
        
        # Get the mapping for this config group
        this_mapping = mapping_df[mapping_df['config_group'] == config_group].copy()
        print(f"Expected {len(this_mapping)} overrides in mapping, found {len(overrides)} in config")
        
        # Create a dict: original_variant -> new_variant for validation
        orig_to_new = {}
        for _, row in this_mapping.iterrows():
            orig_to_new[row['original_variant']] = row['new_variant']
        
        # Check each override
        errors = []
        warnings = []
        
        for i, override in enumerate(overrides):
            variant_num = override.get('variant')
            initial_state_file = override.get('initial_state_file', '')
            
            # Extract original variant from path
            path_match = re.search(r'variant=(\d+)/', initial_state_file)
            if not path_match:
                warnings.append(f"  Override {i}: Could not extract variant from path: {initial_state_file}")
                continue
            
            original_variant_from_path = int(path_match.group(1))
            expected_new_variant = orig_to_new.get(original_variant_from_path)
            
            # Validate
            if variant_num != expected_new_variant:
                errors.append({
                    'index': i,
                    'override_variant': variant_num,
                    'path_original_variant': original_variant_from_path,
                    'expected_new_variant': expected_new_variant,
                    'lineage_seed': override.get('lineage_seed')
                })
        
        # Report findings
        if errors:
            print(f"❌ ERRORS found: {len(errors)} override(s) have mismatched variant numbers")
            for e in errors[:5]:  # Show first 5
                print(f"   Index {e['index']}: override.variant={e['override_variant']} but should be {e['expected_new_variant']}")
                print(f"               (path has original variant={e['path_original_variant']}, seed={e['lineage_seed']})")
            if len(errors) > 5:
                print(f"   ... and {len(errors)-5} more")
        else:
            print(f"✓ All override variant numbers are correct!")
        
        if warnings:
            print(f"⚠ Warnings: {len(warnings)}")
            for w in warnings[:3]:
                print(w)


Loaded mapping CSV: 553 rows


Validating: N_gene_knockout_TU_ID_resume_add_4_to8.json
Expected 0 overrides in mapping, found 1 in config
❌ ERRORS found: 1 override(s) have mismatched variant numbers
   Index 0: override.variant=1 but should be None
               (path has original variant=102, seed=101)

Validating: N_gene_knockout_TU_ID_resume_add_5_to8.json
Expected 0 overrides in mapping, found 476 in config
❌ ERRORS found: 476 override(s) have mismatched variant numbers
   Index 0: override.variant=64 but should be None
               (path has original variant=76, seed=101)
   Index 1: override.variant=64 but should be None
               (path has original variant=76, seed=100)
   Index 2: override.variant=91 but should be None
               (path has original variant=106, seed=101)
   Index 3: override.variant=91 but should be None
               (path has original variant=106, seed=100)
   Index 4: override.variant=196 but should be None
               (path has original var